# Smart Attendance AI — Run in Colab
Runs the full FastAPI backend (RetinaFace detection + DINOv2 ViT recognition + FLAN-T5 report generation) and serves the React frontend from the same server, tunneled publicly via ngrok so you can open it like a real website.

**Before starting:** Runtime → Change runtime type → GPU (T4).

**Get the project onto Colab** — pick ONE of the two cells below.

### Option A — clone from your GitHub repo (recommended once you've pushed it)

In [ ]:
# Replace with your repo URL after you've pushed this project to GitHub
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/smart-attendance-app.git"

!git clone $GITHUB_REPO_URL /content/smart-attendance-app
%cd /content/smart-attendance-app

### Option B — upload the project as a zip instead (skip Option A if you use this)

In [ ]:
from google.colab import files
import zipfile, os

print('Upload smart-attendance-app.zip ...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')
%cd /content/smart-attendance-app

## 1. Install backend dependencies

In [ ]:
%cd /content/smart-attendance-app/backend
!pip install -r requirements.txt -q
!pip uninstall -y onnxruntime -q
!pip install onnxruntime-gpu -q

In [ ]:
import onnxruntime as ort
print('Providers:', ort.get_available_providers())
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Set up an ngrok tunnel (free — sign up at ngrok.com, copy your authtoken)

In [ ]:
from pyngrok import ngrok, conf

# Paste your free ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
conf.get_default().auth_token = NGROK_AUTH_TOKEN

ngrok.kill()  # clear any old tunnels
public_url = ngrok.connect(8000)
print(f'\n🌐 Your app is live at: {public_url}\n')

## 3. Launch the server
This cell blocks (keeps running) while the app is live — that's expected. Open the ngrok URL printed above in your browser. Stop the cell to shut the server down.

In [ ]:
import subprocess, sys

# Runs uvicorn as a subprocess so model loading logs stream live below.
# First run downloads: RetinaFace (~350MB), DINOv2-base (~330MB), FLAN-T5-base (~250MB).
process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='')

## Notes
- **`attendance.db`** (SQLite) is created in the `backend/` folder and persists roster + session history for the life of this Colab runtime. Download it (or copy to Drive) if you want to keep it between sessions.
- **Model downloads are cached** by HuggingFace/insightface after the first run, so subsequent restarts within the same session are faster.
- **To push to GitHub**: from your local machine or a Colab terminal, `git init`, add a `.gitignore` (already included), commit, and push. Don't commit `attendance.db` or `node_modules`.
- **Free ngrok** tunnels show an interstitial warning page on first visit — click "Visit Site" to proceed. This is normal for the free tier.